## 6. Transfer stress test: CODA and MUB6 hosts

The same lesson about refusing unlicensed evidence can be *tested* in
two other source trees. CODA's finite-QM campaign checks whether a
result matches a frozen schema; that does not check whether its values
are true. MUB6's packet firewall refuses an undeclared operator and a
forbidden carrier codepoint; that does not classify MUB(6). These are
distinct programs and distinct contracts. This exercise offers no
Alpha-to-CODA or Alpha-to-MUB6 semantics.

**Assessment D (3 points).** Predict which CODA result is rejected and
which two MUB6 inputs refuse. Run the source-backed checks, then explain
why a clean schema and a clean packet can still carry a false claim.


In [ ]:
import json
import os
import sys
import tempfile
from hashlib import sha256
from pathlib import Path


def find_repositories():
    candidates = [Path.cwd(), *Path.cwd().parents]
    extra = os.environ.get("GENESIS_REPOS_HOME")
    if extra:
        candidates.insert(0, Path(extra))
    for root in candidates:
        has_genesis = (root / "genesis" / "3_alpha_mathematical_language" / "Cargo.toml").is_file() or (root / "3_alpha_mathematical_language" / "Cargo.toml").is_file()
        has_symbol = (root / "the_symbol_game" / "omega_engine_education").is_dir() or (root / "omega_engine_education").is_dir()
        if has_genesis and has_symbol:
            g_root = root / "genesis" if (root / "genesis").is_dir() else root
            s_root = root / "the_symbol_game" if (root / "the_symbol_game").is_dir() else root
            return g_root, s_root
    raise RuntimeError(
        "Place genesis and the_symbol_game beside each other or set GENESIS_REPOS_HOME"
    )


GENESIS, SYMBOL = find_repositories()
REPOS_HOME = GENESIS.parent
ALPHA = Path(os.environ.get("GENESIS_ALPHA_ROOT", GENESIS / "3_alpha_mathematical_language"))
_PFP_PAPERS = (
    SYMBOL
    / "omega_engine_atlas"
    / "coda_atlas"
    / "F004_papers"
    / "015_pfp"
)
PAPER33 = _PFP_PAPERS / "005_pfp_practitioners_field_manual" / "code"
PAPER32 = _PFP_PAPERS / "004_pfp_native_verification_and_evidence_custody" / "code"


def digest(path):
    return sha256(Path(path).read_bytes()).hexdigest()


source_id = {
    "alpha_front_end_sha256": digest(ALPHA / "src" / "lib.rs"),
    "paper032_governor_sha256": digest(PAPER32 / "l1_symbol_game_governor.py"),
    "paper033_toolkit_sha256": digest(PAPER33 / "symbol_game_toolkit.py"),
}
print(json.dumps(source_id, indent=2))


In [ ]:
import importlib.util

coda_checker = (REPOS_HOME / "CODA" / "apps" / "finite_qm" / "campaigns" / "check_result_schema.py" if (REPOS_HOME / "CODA" / "apps" / "finite_qm" / "campaigns" / "check_result_schema.py").is_file() else SYMBOL / "omega_engine_projects" / "finite_qm" / "campaigns" / "check_result_schema.py")
MUB6_CODE = (
    SYMBOL / "research" / "mssn_sn_tower" / "papers" / "012_mutually_unbiased_bases" / "code"
    if (SYMBOL / "research" / "mssn_sn_tower" / "papers" / "012_mutually_unbiased_bases" / "code").is_dir()
    else SYMBOL / "omega_engine_projects" / "mub6" / "code"
)
mub_firewall = MUB6_CODE / "mub_verify" / "packet_firewall.py"
if not coda_checker.is_file() or not mub_firewall.is_file():
    raise RuntimeError(
        "the CODA sibling checkout and the MUB6 paper code are required for the domain-host exercise"
    )
spec = importlib.util.spec_from_file_location("coda_schema_lesson", coda_checker)
coda_schema = importlib.util.module_from_spec(spec)
spec.loader.exec_module(coda_schema)
if str(MUB6_CODE) not in sys.path:
    sys.path.insert(0, str(MUB6_CODE))
from mub_verify.packet_firewall import FirewallRefusalCode, PacketFirewallHost

with tempfile.TemporaryDirectory(prefix="schema-practicum-") as tmp:
    plan = Path(tmp) / "PLAN.md"
    result = Path(tmp) / "result.json"
    plan.write_text(
        '## Result schema\n```json\n{"status": "string", "open_fields": {"field": "reason"}}\n```\n',
        encoding="utf-8",
    )
    result.write_text(json.dumps({"status": "OPEN", "open_fields": {}}), encoding="utf-8")
    clean_schema = coda_schema.check(plan, result)
    result.write_text(
        json.dumps({"status": "OPEN", "open_fields": {}, "generated_at": "forged"}),
        encoding="utf-8",
    )
    dirty_schema = coda_schema.check(plan, result)
assert clean_schema == []
assert any("NON-DETERMINISTIC" in item for item in dirty_schema)

firewall = PacketFirewallHost()
allowed, operator = firewall.lint_first_token_op("ASSERT alpha")
rejected, reason = firewall.lint_first_token_op("QED alpha")
carrier_refusal = firewall.check_fw_carrier("ASSERT alpha \x07 bell")
assert allowed and operator == "ASSERT"
assert not rejected and "undeclared" in reason.lower()
assert not carrier_refusal.admitted
assert carrier_refusal.refusal_code == FirewallRefusalCode.FW_CARRIER_VIOLATION
print("CODA schema:", clean_schema, dirty_schema)
print("MUB6 firewall:", operator, reason, carrier_refusal.refusal_code.value)
print("source digests:", digest(coda_checker), digest(mub_firewall))
score_d = 3
print("D:", score_d, "/3")

## Answer key and evidence ceiling

<details><summary>Open after attempting A–C</summary>

- A: `(2 data, 1 form, 6 events)`. `supply`, admission, and imposition
  are events; observations are diagnostics. Adding `check data` leaves
  the event count six.
- B: the supplied mutant refuses on line 4 with arity 2 versus 1.
  Deleting `admit instance seen a` leaves the positive derivability
  check without its witness.
- M: the first and second repair each emit a new form and instance;
  the inert checkpoint emits none. Reports can agree after regeneration
  even though states and histories differ. The cookie labels are an
  outside interpretation of uninterpreted tokens.
- C: the adapter refuses a changed source, forged output, absent map,
  and absent diagnostic, while the host's
  self-edge violates its irreflexivity rule. The one-edge zero is
  board-relative. The weighted cycle starts with `Phi_1=3` and reaches
  zero only by replacing its claims.
- D: CODA accepts the conforming field set and flags the added
  timestamp field. MUB6 accepts `ASSERT` but refuses `QED` and a
  control character. These are schema and packet checks, respectively.

</details>

**19/19 local exercise points** are available from A, B, M, C, and D
(D contributes three points when the section 6 checks pass). They check
source-backed execution on this machine. The missing work is a typed,
observation-preserving Alpha-to-Omega interpretation and independent
evidence that the corresponding host is the intended Omega Engine.
Source anchors: Alpha `src/lib.rs`, `examples/first.alpha`,
`examples/refused.alpha`; Symbol Game Paper 032
`code/l1_symbol_game_governor.py`; Paper 033
`code/symbol_game_toolkit.py`. The first cell prints current SHA-256
digests for those executable sources. The last code cell prints SHA-256
digests for CODA's `check_result_schema.py` and MUB6's
`packet_firewall.py`.


> **Claim-typing legend**
> - **[Pinned]**: taken from a file this repository already has.
> - **[Proved]**: a finite check in this notebook. Not a general proof.
> - **[Declared]**: a carrier, rule, or heading chosen for this sitting.
> - **[Open]**: something this sitting does not decide.
## Anchors

- Former notebook: `13_alpha_to_omega_execution`
- The code checks one finite example. It is not a general proof.

## Exercise

Change one declared input in the code cell. The check should fail for the wrong object and pass for the finite example written here.
